In [ ]:
#pip install requests pandas openpyxl xlrd ipykernel

Imports y configuración

In [4]:
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def encontrar_raiz(inicio=None) -> Path:
    """Sube por el árbol de carpetas hasta encontrar la raíz del proyecto."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        tiene_data = (carpeta / "data").is_dir()
        tiene_marca = any((carpeta / m).exists() for m in (".git", "Makefile", "pyproject.toml"))
        if tiene_data and tiene_marca:
            return carpeta
    raise FileNotFoundError("No encontré la raíz del proyecto (carpeta con data/ y .git/Makefile/pyproject.toml).")


RAIZ = encontrar_raiz()

# Ruta relativa a la raíz del proyecto. Cámbiala cuando me digas la definitiva.
RUTA_DESTINO = Path("data/raw")
DESTINO = RAIZ / RUTA_DESTINO
DESTINO.mkdir(parents=True, exist_ok=True)

# --- Fuente 1: URL directa al Excel 1990-2015 ---
URL_1990_2015 = (
    "https://coespo.sonora.gob.mx/documentos/Marginacion/Municipal/"
    "Indice-de-Marginacion-municipal-1990-2015(Conapo).xlsx"
)

# --- Fuente 2: portal CKAN, recurso 2020 ---
CKAN_BASE = "https://datos.sonora.gob.mx"
CKAN_RESOURCE_ID = "046c2f52-72aa-4059-bf05-5ad52e191d1b"

# Plan B: si la API falla, pega aquí la URL directa del archivo 2020.
URL_2020_MANUAL = "https://datos.sonora.gob.mx/dataset/%C3%8Dndice%20de%20marginaci%C3%B3n%20por%20municipio/download_resource/046c2f52-72aa-4059-bf05-5ad52e191d1b"

print(f"Raíz del proyecto: {RAIZ}")
print(f"Carpeta de destino: {DESTINO}")

Raíz del proyecto: C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda
Carpeta de destino: C:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda\data\raw


Sesión HTTP 

In [2]:
def crear_sesion() -> requests.Session:
    """Sesión con User-Agent de navegador y reintentos automáticos."""
    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept": "*/*",
    })
    reintentos = Retry(
        total=4,
        backoff_factor=1.5,                      # espera 0s, 1.5s, 3s, 6s...
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    sesion.mount("https://", HTTPAdapter(max_retries=reintentos))
    sesion.mount("http://", HTTPAdapter(max_retries=reintentos))
    return sesion


sesion = crear_sesion()

Resolver URL de 2020 vía API de CKAN

In [5]:
def url_directa_ckan(base: str, resource_id: str, sesion: requests.Session) -> str:
    """Pregunta a la API de CKAN cuál es la URL real de descarga de un recurso."""
    resp = sesion.get(
        f"{base}/api/3/action/resource_show",
        params={"id": resource_id},
        timeout=30,
    )
    resp.raise_for_status()
    datos = resp.json()
    if not datos.get("success"):
        raise RuntimeError(f"CKAN respondió sin éxito: {datos}")
    return datos["result"]["url"]


if URL_2020_MANUAL:
    URL_2020 = URL_2020_MANUAL
else:
    try:
        URL_2020 = url_directa_ckan(CKAN_BASE, CKAN_RESOURCE_ID, sesion)
    except Exception as e:
        raise RuntimeError(
            "No pude obtener la URL por la API. Copia la dirección del enlace de descarga "
            "en el navegador y pégala en URL_2020_MANUAL (Celda 1)."
        ) from e

print("URL 2020:", URL_2020)

URL 2020: https://datos.sonora.gob.mx/dataset/%C3%8Dndice%20de%20marginaci%C3%B3n%20por%20municipio/download_resource/046c2f52-72aa-4059-bf05-5ad52e191d1b


Función de descarga

In [6]:
# "Firmas" (primeros bytes) de los formatos de Excel, para validar el archivo descargado
FIRMA_XLSX = b"PK"                                   # .xlsx es un zip
FIRMA_XLS = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"      # .xls antiguo (OLE2)


def descargar_archivo(url: str, destino: Path, sesion: requests.Session,
                      sobrescribir: bool = False) -> Path:
    """Descarga `url` a `destino` por bloques y verifica que sea un Excel real."""
    if destino.exists() and not sobrescribir:
        print(f"Ya existe, se omite: {destino.name}")
        return destino

    temporal = destino.with_suffix(destino.suffix + ".part")
    with sesion.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(temporal, "wb") as f:
            for bloque in resp.iter_content(chunk_size=1024 * 256):
                f.write(bloque)

    with open(temporal, "rb") as f:
        inicio = f.read(8)
    if not (inicio.startswith(FIRMA_XLSX) or inicio.startswith(FIRMA_XLS)):
        temporal.unlink()
        raise ValueError(
            f"Lo descargado de {url} no parece un Excel (¿página HTML de error o bloqueo?)."
        )

    temporal.replace(destino)  # renombra solo si todo salió bien
    print(f"Descargado: {destino.name} ({destino.stat().st_size / 1024:.1f} KB)")
    return destino

5. Ejecutar las descargas

In [7]:
# Para 2020 conservamos la extensión original (.xls o .xlsx) según la URL
ext_2020 = Path(URL_2020.split("?")[0]).suffix.lower() or ".xlsx"

FUENTES = {
    "indice_marginacion_municipal_1990_2015.xlsx": URL_1990_2015,
    f"indice_marginacion_municipal_2020{ext_2020}": URL_2020,
}

rutas = {}
for nombre, url in FUENTES.items():
    rutas[nombre] = descargar_archivo(url, DESTINO / nombre, sesion)

Descargado: indice_marginacion_municipal_1990_2015.xlsx (2137.5 KB)
Descargado: indice_marginacion_municipal_2020.xlsx (24.0 KB)


6. Validacion con pandas

In [9]:
# Hoja de datos que nos interesa de cada archivo
HOJAS_A_CARGAR = {
    "indice_marginacion_municipal_1990_2015.xlsx": "Base_Indice_de_marginacion_muni",
    f"indice_marginacion_municipal_2020{ext_2020}": "IMM_2020",
}

dataframes = {}

for nombre, hoja in HOJAS_A_CARGAR.items():
    df = pd.read_excel(rutas[nombre], sheet_name=hoja)
    dataframes[nombre] = df

    print("=" * 70)
    print(f"{nombre} | hoja '{hoja}': {df.shape[0]} filas x {df.shape[1]} columnas")
    print("Columnas:", list(df.columns))
    display(df.head(3))

indice_marginacion_municipal_1990_2015.xlsx | hoja 'Base_Indice_de_marginacion_muni': 14646 filas x 24 columnas
Columnas: ['CVE_MUN', 'CVE_ENT', 'ENT', 'CVEE_MUN', 'MUN', 'POB_TOT', 'VP', 'ANALF', 'SPRIM', 'OVSDE', 'OVSEE', 'OVSAE', 'VHAC', 'OVPT', 'PL<5000', 'PO2SM', 'OVSD', 'OVSDSE', 'IM', 'GM', 'IND0A100', 'LUGAR_NAC', 'LUGAR_EST', 'AÑO']


,CVE_MUN,CVE_ENT,ENT,CVEE_MUN,MUN,POB_TOT,VP,ANALF,SPRIM,OVSDE,...,PL<5000,PO2SM,OVSD,OVSDSE,IM,GM,IND0A100,LUGAR_NAC,LUGAR_EST,AÑO
0,1001,1,Aguascalientes,1,Aguascalientes,506 274,-,6.05,27.99,6.55,...,11.47,58.36,-,-,-1.833,Muy bajo,-,2 341,9,1990
1,1002,1,Aguascalientes,2,Asientos,32 225,-,11.02,53.64,41.21,...,100,82.06,-,-,-0.383,Medio,-,1 532,1,1990
2,1003,1,Aguascalientes,3,Calvillo,48 440,-,9.56,53.65,13.89,...,54.34,72.01,-,-,-0.957,Bajo,-,1 953,7,1990


indice_marginacion_municipal_2020.xlsx | hoja 'IMM_2020': 72 filas x 18 columnas
Columnas: ['CVE_ENT', 'NOM_ENT', 'CVE_MUN', 'NOM_MUN', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM_2020', 'GM_2020', 'IMN_2020', 'POSICION']


,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM_2020,GM_2020,IMN_2020,POSICION
0,26,Sonora,26001,Aconchi,2563,2.691722,37.125444,0.780336,0.858369,0.702302,0.312134,16.329113,100.000000,66.869301,57.175873,Muy bajo,0.895295,1961
1,26,Sonora,26002,Agua Prieta,91929,1.488203,26.019518,0.203682,1.035895,0.618724,0.946196,17.610458,0.979016,81.065659,58.632120,Muy bajo,0.918097,2267
2,26,Sonora,26003,Alamos,24976,8.107527,41.656634,7.845339,4.087117,5.190117,10.769724,26.759377,56.113869,75.732267,53.436432,Medio,0.836740,996


In [10]:
df_1990_2015 = dataframes["indice_marginacion_municipal_1990_2015.xlsx"]
df_sonora_1990_2015 = df_1990_2015[df_1990_2015["CVE_ENT"] == 26].copy()

print(df_sonora_1990_2015.shape)
print(df_sonora_1990_2015["AÑO"].value_counts().sort_index())

(428, 24)
AÑO
1990    70
1995    70
2000    72
2005    72
2010    72
2015    72
Name: count, dtype: int64
